In [5]:
from pathlib import Path
import sys


for candidate in (
    Path.cwd(),
    *Path.cwd().parents,
):
    if (
        candidate
        / "pyproject.toml"
    ).is_file():
        project_root = candidate
        break
else:
    raise FileNotFoundError(
        "Could not find the repository root. "
        "Expected a parent containing "
        "pyproject.toml."
    )


seed = 1


SELECTED_FILENAMES = [
    (
        "400-1293-17-c11k1r5_pd_"
        "sp1_2_fl0_1_right.tif"
    ),
    (
        "400-1293-17-c11k1r5_pd_"
        "sp1_2_fl0_1_left.tif"
    ),
    (
        "400-1293-w14_c05k1r6_sp0_2_"
        "flow0_1_top_right.tif"
    ),
    (
        "400-1093-w11-c1p1-befo3-"
        "10sp-15flow_right.tif"
    ),
    (
        "400-1605-W7_map_00021_4.tif"
    ),
]


notebook_dir = (
    project_root
    / "notebooks"
    / "review"
    / "R4.4_gradual_noise"
)

sys.path.insert(
    0,
    str(notebook_dir),
)


from noise_degradation import (
    build_noisy_datasets,
    plot_degradation_examples,
)


path_to_images = (
    project_root
    / "data"
    / "cnt_segmentation"
    / "images"
)

path_to_synthetic_noise = (
    project_root
    / "data"
    / "cnt_segmentation"
    / "synthetic_noisy_images"
)


path_to_synthetic_noise.mkdir(
    parents=True,
    exist_ok=True,
)


print(
    "Selected images:",
    len(SELECTED_FILENAMES),
)

for filename in SELECTED_FILENAMES:
    print(filename)

Selected images: 5
400-1293-17-c11k1r5_pd_sp1_2_fl0_1_right.tif
400-1293-17-c11k1r5_pd_sp1_2_fl0_1_left.tif
400-1293-w14_c05k1r6_sp0_2_flow0_1_top_right.tif
400-1093-w11-c1p1-befo3-10sp-15flow_right.tif
400-1605-W7_map_00021_4.tif


In [6]:
# Build the synthetic noisy variants and save the example figure.
noise_output_records = (
    build_noisy_datasets(
        path_to_images,
        path_to_synthetic_noise,
        selected_filenames=(
            SELECTED_FILENAMES
        ),
        seed=seed,
    )
)


plot_degradation_examples(
    path_to_images,
    path_to_synthetic_noise,
    (
        path_to_synthetic_noise
        / "noise_degradation_examples.png"
    ),
    selected_filenames=(
        SELECTED_FILENAMES
    ),
    seed=seed,
)

Generated noise for 5 selected images.
Severity stages: 4
Generated TIFF files: 60
Saved degradation visualization: c:\Users\abd93000\PycharmProjects\cnt_project_v2\data\cnt_segmentation\synthetic_noisy_images\noise_degradation_examples.png
Displayed source image: 400-1293-17-c11k1r5_pd_sp1_2_fl0_1_left.tif


In [7]:
from pathlib import Path

import tifffile


original_root = (
    project_root
    / "data"
    / "cnt_segmentation"
    / "images"
)

noisy_root = (
    project_root
    / "data"
    / "cnt_segmentation"
    / "synthetic_noisy_images"
)


for noise_directory in [
    "gaussian_noise",
    "salt_and_pepper",
    "both_added",
]:
    output_paths = sorted(
        (
            noisy_root
            / noise_directory
        ).glob("*.tif")
    )

    shapes = {
        tifffile.imread(path).shape
        for path in output_paths
    }

    dtypes = {
        str(tifffile.imread(path).dtype)
        for path in output_paths
    }

    print(
        noise_directory,
        "| files:",
        len(output_paths),
        "| shapes:",
        shapes,
        "| dtypes:",
        dtypes,
    )

gaussian_noise | files: 0 | shapes: set() | dtypes: set()
salt_and_pepper | files: 0 | shapes: set() | dtypes: set()
both_added | files: 0 | shapes: set() | dtypes: set()


In [9]:
import tifffile
import pandas as pd


EXPECTED_STAGES = [
    1,
    2,
    3,
    4,
]

EXPECTED_NOISE_TYPES = [
    "gaussian_noise",
    "salt_and_pepper",
    "both_added",
]


validation_rows = []


for stage in EXPECTED_STAGES:
    for noise_type in (
        EXPECTED_NOISE_TYPES
    ):
        output_directory = (
            path_to_synthetic_noise
            / f"stage_{stage}"
            / noise_type
        )

        output_paths = sorted(
            output_directory.glob(
                "*.tif"
            )
        )

        output_names = {
            path.name
            for path in output_paths
        }

        expected_names = set(
            SELECTED_FILENAMES
        )

        shapes = {
            tifffile.imread(
                path
            ).shape
            for path in output_paths
        }

        dtypes = {
            str(
                tifffile.imread(
                    path
                ).dtype
            )
            for path in output_paths
        }

        status = (
            "PASS"
            if (
                len(output_paths) == 5
                and output_names
                == expected_names
            )
            else "FAIL"
        )

        validation_rows.append(
            {
                "stage": stage,
                "noise_type": (
                    noise_type
                ),
                "file_count": len(
                    output_paths
                ),
                "shapes": str(
                    sorted(shapes)
                ),
                "dtypes": str(
                    sorted(dtypes)
                ),
                "status": status,
            }
        )


validation_df = pd.DataFrame(
    validation_rows
)


display(
    validation_df
)


if not validation_df[
    "status"
].eq("PASS").all():
    raise RuntimeError(
        "One or more generated noise "
        "directories failed validation."
    )


print(
    "PASS: all 4 stages contain all "
    "3 noise conditions and exactly "
    "5 selected images."
)

print(
    "Total generated TIFFs:",
    validation_df[
        "file_count"
    ].sum(),
)

,stage,noise_type,file_count,shapes,dtypes,status
0,1,gaussian_noise,5,"[(256, 256, 3)]",['uint8'],PASS
1,1,salt_and_pepper,5,"[(256, 256, 3)]",['uint8'],PASS
2,1,both_added,5,"[(256, 256, 3)]",['uint8'],PASS
3,2,gaussian_noise,5,"[(256, 256, 3)]",['uint8'],PASS
4,2,salt_and_pepper,5,"[(256, 256, 3)]",['uint8'],PASS
5,2,both_added,5,"[(256, 256, 3)]",['uint8'],PASS
6,3,gaussian_noise,5,"[(256, 256, 3)]",['uint8'],PASS
7,3,salt_and_pepper,5,"[(256, 256, 3)]",['uint8'],PASS
8,3,both_added,5,"[(256, 256, 3)]",['uint8'],PASS
9,4,gaussian_noise,5,"[(256, 256, 3)]",['uint8'],PASS


PASS: all 4 stages contain all 3 noise conditions and exactly 5 selected images.
Total generated TIFFs: 60


import tiff